# Dengue Preprocessing
Agregação por UF, exclusão do Espírito Santo e normalização de casos pela população.

In [ ]:
import pandas as pd
import numpy as np

## 1. Carregamento dos dados

In [ ]:
dengue = pd.read_csv('../DataRaw/dengue.csv', parse_dates=['date'])
population = pd.read_csv('../DataRaw/datasus_population_2001_2025.csv')

print('dengue:', dengue.shape)
print('population:', population.shape)
dengue.head()

## 2. Exclusão do Espírito Santo

In [ ]:
dengue = dengue[dengue['uf'] != 'ES'].copy()
print('Após exclusão de ES:', dengue.shape)

## 3. Join com população (nível município)

In [ ]:
dengue['year'] = dengue['date'].dt.year

dengue = dengue.merge(
    population[['geocode', 'year', 'population']],
    on=['geocode', 'year'],
    how='left'
)

missing_pop = dengue['population'].isna().sum()
print(f'Municípios sem população correspondente: {missing_pop} linhas ({missing_pop/len(dengue)*100:.1f}%)')

## 4. Agregação por UF

In [ ]:
TRAIN_TEST_COLS = ['train_1', 'target_1', 'train_2', 'target_2',
                   'train_3', 'target_3', 'train_4', 'target_4']

agg_dict = {'casos': 'sum', 'population': 'sum'}
# train/test são consistentes dentro de UF+data — basta pegar o primeiro valor
agg_dict.update({col: 'first' for col in TRAIN_TEST_COLS})

dengue_uf = (
    dengue
    .groupby(['date', 'epiweek', 'uf', 'uf_code', 'year'], as_index=False)
    .agg(agg_dict)
)

print('Shape após agregação por UF:', dengue_uf.shape)
dengue_uf.head()

## 5. Normalização: incidência por 100.000 habitantes

In [ ]:
dengue_uf['casos_norm'] = (dengue_uf['casos'] / dengue_uf['population']) * 100_000

dengue_uf['casos_norm'].describe()

## 6. Verificação final

In [ ]:
print('UFs presentes:', sorted(dengue_uf['uf'].unique()))
print('\nPeriodo:', dengue_uf['date'].min(), 'a', dengue_uf['date'].max())
print('\nColunas finais:')
print(dengue_uf.dtypes)

## 7. Exportação

In [ ]:
output_cols = ['date', 'epiweek', 'year', 'uf', 'uf_code',
               'casos', 'population', 'casos_norm'] + TRAIN_TEST_COLS

dengue_uf[output_cols].to_csv('dengue_uf_preprocessed.csv', index=False)
print('Salvo em: 1_Preprocessing/dengue_uf_preprocessed.csv')